In [1]:
import torch
from models.GPTModel import GPTModel
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "num_layers": 12,
    "num_heads": 12,
    "emb_dim": 768,
    "context_length": 256,
    "dropout": 0.1,
    "num_classes": 2,
    'bias':False
}
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_embedding): Embedding(50257, 768)
  (pos_embedding): Embedding(256, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attention): MultiHeadAttention(
        (w_q): Linear(in_features=768, out_features=768, bias=False)
        (w_k): Linear(in_features=768, out_features=768, bias=False)
        (w_v): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ffn): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attention): MultiHeadAttention(
        (w_q): Linear(in_feat

In [2]:
import tiktoken

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    decoded = tokenizer.decode(token_ids.squeeze(0).tolist())
    return decoded

def generate_text(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
        
    return idx

In [3]:
context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = generate_text(
    model=model,
    idx=text_to_token_ids(context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M['context_length']
)
print(token_ids_to_text(token_ids, tokenizer))

Every effort moves you rentingetic wasnم refres RexMeCHicular stren


In [4]:
inputs = torch.tensor([[16833, 3626, 6100],
                       [40, 1107, 588]])
targets = torch.tensor([[3626, 6100, 345],
                        [1107, 588, 11311]])
with torch.no_grad():
    logits = model(inputs)
probas = torch.softmax(logits, dim=-1)
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
target_probas_2 = probas[1, [0, 1, 2], targets[1]]

In [5]:
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2), dim=-1))
print(log_probas)

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])


In [6]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

tensor(10.7940)


## 加载数据集

In [7]:
file_path = "the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()
train_ratio = 0.9
split_idx = int(len(text) * train_ratio)
train_data = text[:split_idx]
val_data = text[split_idx:]

In [8]:
print(len(train_data), len(val_data))

18431 2048


In [10]:
from datasets.dataloader import create_dataloader
torch.manual_seed(123)
train_loader = create_dataloader(train_data, batch_size=2, max_length=GPT_CONFIG_124M['context_length'], 
                                 stride=GPT_CONFIG_124M['context_length'], shuffle=True, drop_last=True, num_workers=0)
val_loader = create_dataloader(val_data, batch_size=2, max_length=GPT_CONFIG_124M['context_length'], 
                               stride=GPT_CONFIG_124M['context_length'], shuffle=False, drop_last=True, num_workers=0)

4612
1
534
1


In [11]:
def calc_loss(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten()
    )
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0
    if len(data_loader) == 0:
        return float('nan')
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)
    valid_loss = calc_loss_loader(val_loader, model, device)
print(f"train_loss: {train_loss:.3f}, valid_loss: {valid_loss:.3f}")

train_loss: 10.988, valid_loss: 10.981


In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, eval_iter)
        valid_loss = calc_loss_loader(val_loader, model, device, eval_iter)
    model.train()
    return train_loss, valid_loss

def generate_and_print_sample(model, tokenizer, start_context, device):
    model.eval()
    context_size = model.pos_embedding.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text(model, encoded, 50, context_size)
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace('\n', ' ')) 
    model.train()
    

def train_model_simple(model, train_loader, val_loader, optimizer, device, 
                       epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, step = 0, -1
    for epoch in range(epochs):
        model.train()
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            loss = calc_loss(inputs, targets, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += inputs.numel()
            step += 1

            if step % eval_freq == 0:
                model.eval()
                with torch.no_grad():
                    train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                    train_losses.append(train_loss)
                    val_losses.append(val_loss)
                    track_tokens_seen.append(tokens_seen)
                    print(f"Epoch: {epoch}, Step: {step}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        generate_and_print_sample(model, tokenizer, start_context, device)
    return train_losses, val_losses, track_tokens_seen, model




In [22]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.1)
epoch = 50
train_losses, val_losses, tokens_seen = train_model_simple(model, train_loader, val_loader, optimizer, device, epoch,
                                                           eval_freq=5, eval_iter=5, start_context="Every effort moves you", tokenizer=tokenizer)

Epoch: 0, Step: 0, Train Loss: 10.5363, Val Loss: 10.5755
Epoch: 0, Step: 5, Train Loss: 9.1990, Val Loss: 9.4168
Every effort moves you,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Epoch: 1, Step: 10, Train Loss: 8.6211, Val Loss: 8.8858
Epoch: 1, Step: 15, Train Loss: 8.1435, Val Loss: 8.4213
Every effort moves you, the, the the the the the the the the the the the.                                   
Epoch: 2, Step: 20, Train Loss: 7.6503, Val Loss: 7.9868
Epoch: 2, Step: 25, Train Loss: 7.1413, Val Loss: 7.5929
Every effort moves you, the, the, the, the, the, the, the.                                   
Epoch: 3, Step: 30, Train Loss: 6.6928, Val Loss: 7.2656
Epoch: 3, Step: 35, Train Loss: 6.3420, Val Loss: 6.9876
Every effort moves you, and, and the the the the the the the the.                                     
Epoch: 4, Step: 40, Train Loss: 5.8218, Val Loss: 6.8166
Every effort moves you, and I had the the of the of the the of the.  ", I had the of the of the of the of th

## 控制随机性的解码策略

In [24]:
model.to('cpu')
model.eval()
token_ids = generate_text(model, text_to_token_ids("Every effort moves you", tokenizer),
                          25,
                          context_size=GPT_CONFIG_124M['context_length'])
print(token_ids_to_text(token_ids, tokenizer))

Every effort moves you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"




In [25]:
def generate(model, idx, max_new_tokens, context_size,
             temperature=1.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        if top_k is not None:
            top_logits, _ = torch.topk(logits, k=top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, 
                                 torch.tensor(-float('Inf'), device=logits.device), 
                                 logits)
        if temperature > 0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.softmax(logits, dim=-1)
        if idx_next == eos_id:
            break
        idx = torch.cat((idx, idx_next), dim=-1)
    return idx

In [29]:
token_ids = generate(
    model, 
    text_to_token_ids("Every effort moves you", tokenizer), 
    max_new_tokens=256,
    context_size=128,
    temperature=1.4,
    top_k=25 
)
print(token_ids_to_text(token_ids, tokenizer))

Every effort moves you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"

He laughed again, and threw back his head to look up at the sketch of the donkey. "There were days when I couldn't look at that thing--couldn't face it. But I forced myself to put it here; and now it's cured me--cured me. That's the reason why I don't dabble any more, my dear Rickham; or rather Stroud himself is the reason."

For the first time my idle curiosity about my companion turned into a eyes. The word Usually; and I have Usually eyes over! Usually I've chucked hand. I said: "Gisburn, I hand, I have been born of the, at the heart me now it. The rest of the hand, Iorid Jack's the last thing whom showy; and I seemed to heart attack. The younger by whom him when up. Gisburn, heart heart a rule, and Usually mine: "Yes--oror I've hand, I was just a kind that areor etching Usually _felt_ watching me queer. I have been taken. I have been me. I had could
